# Lab 5 — Part 5: Create scoped resources and run the managed pipeline

**พาร์ตนี้เริ่มใช้งาน Azure จริงและมีค่าใช้จ่าย** ใช้ WSL/kernel `itcs355-lab3` เดิม

ก่อนรัน อ่านผลกระทบ:
- สร้าง Git commit เฉพาะ README/source/tests ของ Lab 5 ในเครื่อง ไม่มี push
- สร้าง identities `itcs355-6688012-lab5-train` และ `itcs355-6688012-lab5-serve`
- ใช้ CI identity `itcs355-6688012-lab4-ci` เดิมบน release compute
- ตรวจ/เพิ่ม role assignments 8 รายการตาม [pipeline-preview.json](../reports/pipeline-preview.json)
  (Storage roles ระดับ container, AcrPull ระดับ ACR และ Managed Identity Operator บน serving identity)
- สร้าง `lab5train` DS2 v2 และ `lab5release` DS1 v2 แบบ min=0/max=1, idle scale-down 120s
- รัน degraded pipeline ก่อน ต้อง fail ที่ evaluate และไม่มี model version/endpoint เพิ่ม
- เมื่อพิสูจน์ fail branch แล้ว รัน normal pipeline ถึง registry, staging และ smoke test
- หากผ่าน จะมี endpoint `itcs355-6688012-lab5` และ model version ใหม่ **endpoint ยังเปิดอยู่หลังจบ**

ราคาที่ตรวจไว้: training/staging DS2 v2 5.6784 THB/hour, release DS1 v2 2.8224 THB/hour
estimate ที่ commit แล้ว: Part A 17.3568 THB, proposed envelope 44.8282 THB (รวมเงินสำรอง ไม่ใช่ hard billing cap)

Budget screening ใช้ **สมมติฐานวางแผน** 40 THB/USD และสำรองเพิ่ม 200 THB สำหรับค่าใช้จ่ายที่ยังไม่ post/นอก scope
ยอดที่ post USD 7.8961 × 40 + 44.8282 + 200 ≈ 560.6710 THB ต่ำกว่า 800 ในสมมติฐานนี้
**40 ไม่ใช่ FX ที่วัดจริง และ 200 ไม่ใช่ค่าใช้จริง** หากทราบค่าใช้จ่ายอื่นที่ทำให้ reserve ไม่พอ ให้หยุดก่อนรัน
ยังไม่อ้างว่าผ่านงบทั้งเทอม; discounted compute ยังไม่ผ่านเพราะ quota รวมเป็น 0

ข้อมูล raw/cloud.env/credentials ไม่ถูก upload กับ code bundle และไม่ถูก commit
notebooks ที่มี outputs กับรายงานการเงินยังไม่ถูก commit ในพาร์ตนี้ เก็บไว้ตรวจงานส่งตอนท้าย

## รัน cell เดียว แล้วส่งผล

มี progress ทุกช่วงและ job status ทุก 15s เวลาส่วนใหญ่อยู่ที่ Azure provisioning
อย่ารัน notebook ซ้อนกัน หากต้องรันใหม่จะใช้ชื่อ job เดิมเพื่อไม่ submit ซ้ำ
หาก error ให้ส่ง output รวม job ID กลับมา **อย่าสร้างงานใหม่เอง**
กด Interrupt หยุดการรอในเครื่องเท่านั้น ไม่ได้ cancel job ใน Azure
หลังผ่านต้องไปขั้นเก็บหลักฐาน/cleanup ต่อ ไม่ควรปล่อย staging ข้ามคืน


In [1]:
from pathlib import Path
import platform
import subprocess
import sys

if platform.system() != "Linux" or "microsoft" not in platform.release().lower():
    raise RuntimeError("เลือก kernel itcs355-lab3 ฝั่ง WSL ก่อนรันครับ")
here = Path.cwd().resolve()
lab = next((base / "lab5" for base in (here, *here.parents)
            if (base / "lab5/cloudlayer/execute.py").is_file()
            and (base / "lab4/README.md").is_file()), None)
if lab is None:
    raise RuntimeError("เปิด notebook จาก repository เดิมก่อนครับ")
print("Part 5: local source commit, scoped Azure setup, and billable managed execution.", flush=True)
with subprocess.Popen(
    [sys.executable, "-u", "-m", "cloudlayer.execute"], cwd=lab,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
) as task:
    try:
        for line in task.stdout:
            print(line, end="", flush=True)
        exit_code = task.wait()
    except KeyboardInterrupt:
        task.terminate()
        try:
            task.wait(timeout=10)
        except subprocess.TimeoutExpired:
            task.kill()
        print("Azure jobs/resources may still be running. Inspect Studio; do not submit another job.", flush=True)
        raise
if exit_code:
    raise RuntimeError("Part 5 หยุด: ส่ง summary/error พร้อม job ID กลับมา อย่าเริ่มงานใหม่ซ้ำครับ")
print("ส่ง SEND THIS SUMMARY กลับมาทันทีครับ staging ยังเปิดอยู่ ต้องเก็บหลักฐานและ cleanup ต่อ", flush=True)


Part 5: local source commit, scoped Azure setup, and billable managed execution.
[14:05:38] 1/5 Verify estimate, budget assumptions and commit implementation source
[14:06:03] Commit 19 allowlisted source files locally; no push
[main 2487815] feat: implement Lab 5 managed training and release pipeline
 19 files changed, 1974 insertions(+)
 create mode 100644 lab5/README.md
 create mode 100644 lab5/cloudlayer/__init__.py
 create mode 100644 lab5/cloudlayer/execute.py
 create mode 100644 lab5/cloudlayer/pipelines.py
 create mode 100644 lab5/cloudlayer/planning.py
 create mode 100644 lab5/cloudlayer/readiness.py
 create mode 100644 lab5/cloudlayer/release.py
 create mode 100644 lab5/cloudlayer/run_step.py
 create mode 100644 lab5/scripts/__init__.py
 create mode 100644 lab5/scripts/billing_retry.py
 create mode 100644 lab5/scripts/plan_run.py
 create mode 100644 lab5/scripts/preflight.py
 create mode 100644 lab5/scripts/prepare_pipeline.py
 create mode 100644 lab5/src/__init__.py
 create 

RuntimeError: Part 5 หยุด: ส่ง summary/error พร้อม job ID กลับมา อย่าเริ่มงานใหม่ซ้ำครับ

## หลักฐานและงานต่อ

เก็บ state/job IDs ที่ `lab5/reports/managed-execution.json`
มี estimate commit, source commit, bundle hashes, identities/roles, child job statuses และ negative-control verification
ชื่อ job ถูกบันทึกก่อน submit เพื่อ resume การติดตาม แทนการสร้างงานใหม่

ยังเหลือ permission-removal experiment, schedule, portability provider ที่สอง, LLM Part B และ final cost/cleanup
พาร์ตนี้จึงยังไม่ใช่ Lab 5 ที่ส่งได้ครบ และยังไม่ลบ endpoint ให้อัตโนมัติ

อ้างอิง: [AmlCompute](https://learn.microsoft.com/en-us/python/api/azure-ai-ml/azure.ai.ml.entities.amlcompute),
[Job operations](https://learn.microsoft.com/en-us/python/api/azure-ai-ml/azure.ai.ml.operations.joboperations),
[Endpoint quota reserve](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-manage-quotas?view=azureml-api-2)
